# 🚗 Vehicle Fuel Efficiency Prediction
## Notebook 3: Data Cleaning

---

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
df = pd.read_csv('../data/auto-mpg.csv', na_values='?')
print(f'Loaded: {df.shape} | Missing in horsepower: {df["horsepower"].isna().sum()}')

Loaded: (398, 9) | Missing in horsepower: 6


## 3.1 Handle Missing Values

In [2]:
print('Rows with missing horsepower:')
display(df[df['horsepower'].isna()])
hp_median = df['horsepower'].median()
df['horsepower'].fillna(hp_median, inplace=True)
print(f'\nImputed missing horsepower with median: {hp_median}')
print(f'Missing values remaining: {df.isnull().sum().sum()}')

Rows with missing horsepower:


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,car_name
32,25.0,4,98.0,NaN,2046,19.0,71,1,ford pinto
126,21.0,6,200.0,NaN,2875,17.0,74,1,ford maverick
330,40.9,4,85.0,NaN,1835,17.3,80,2,renault lecar deluxe
336,23.6,4,140.0,NaN,2905,14.3,80,1,ford mustang cobra
354,34.5,4,100.0,NaN,2320,15.8,81,2,renault 18i
374,23.0,4,151.0,NaN,3035,20.5,82,1,amc concord dl



Imputed missing horsepower with median: 93.5
Missing values remaining: 0


## 3.2 Correct Data Types

In [3]:
df['horsepower'] = df['horsepower'].astype(float)
df['origin'] = df['origin'].astype('category')
print('Updated dtypes:')
print(df.dtypes)

Updated dtypes:
mpg              float64
cylinders          int64
displacement     float64
horsepower       float64
weight             int64
acceleration     float64
model_year         int64
origin          category
car_name          object
dtype: object


## 3.3 Detect & Handle Outliers

In [4]:
numeric_cols = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='#4C72B0', alpha=0.7),
                    medianprops=dict(color='red', linewidth=2),
                    flierprops=dict(marker='o', color='orange', markersize=6))
    axes[i].set_title(f'{col}', fontsize=12, fontweight='bold')
    axes[i].set_xticks([])
plt.suptitle('Outlier Detection — Box Plots', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../plots/02_outlier_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

<Figure size 1600x900 with 6 Axes>

In [5]:
outlier_report = {}
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_report[col] = {'lower_bound': round(lower, 2),
                            'upper_bound': round(upper, 2),
                            'n_outliers': n_outliers}
outlier_df = pd.DataFrame(outlier_report).T
print('IQR Outlier Summary:')
display(outlier_df)
print('\n✅ Decision: Retaining outliers (tree models handle them natively)')

IQR Outlier Summary:


,lower_bound,upper_bound,n_outliers
mpg,0.25,46.25,1.0
cylinders,-2.00,14.00,0.0
displacement,-132.38,498.62,0.0
horsepower,2.50,198.50,11.0
weight,147.38,5684.38,0.0
acceleration,8.80,22.20,7.0



✅ Decision: Retaining outliers (tree models handle them natively)


## 3.4 Remove Redundant Features

In [6]:
df['brand'] = df['car_name'].apply(lambda x: str(x).strip().split()[0].lower())
brand_map = {
    'chevroelt': 'chevrolet',
    'chevy': 'chevrolet',
    'toyouta': 'toyota',
    'maxda': 'mazda',
    'vokswagen': 'volkswagen',
    'vw': 'volkswagen',
}
df['brand'] = df['brand'].replace(brand_map)
print(f'Top 15 brands:')
print(df['brand'].value_counts().head(15))
df.drop(columns=['car_name'], inplace=True)
print('\nDropped car_name, kept brand ✓')

Top 15 brands:
brand
ford          51
chevrolet     47
plymouth      31
amc           28
dodge         28
toyota        26
datsun        23
volkswagen    22
buick         17
pontiac       16
honda         13
mazda         12
mercury       11
oldsmobile    10
fiat           8
Name: count, dtype: int64

Dropped car_name, kept brand ✓


In [7]:
df.to_csv('../data/auto-mpg-cleaned.csv', index=False)
print(f'Cleaned dataset saved: {df.shape}')
df.head()

Cleaned dataset saved: (398, 9)


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,brand
0,18.0,8,307.0,130.0,3504,12.0,70,1,chevrolet
1,15.0,8,350.0,165.0,3693,11.5,70,1,buick
2,18.0,8,318.0,150.0,3436,11.0,70,1,plymouth
3,16.0,8,304.0,150.0,3433,12.0,70,1,amc
4,17.0,8,302.0,140.0,3449,10.5,70,1,ford
